# Customer RFM Segmentation Analysis
## E-Commerce Customer Analytics & Retention Intelligence

### Overview & Objectives
This notebook conducts a complete **Recency, Frequency, and Monetary (RFM) Segmentation Analysis** for our e-commerce platform. RFM is an established, high-impact behavioral segmentation technique that groups customers based on their historical purchasing behavior:

1. **Recency (R)**: How recently did the customer make a purchase? (Days elapsed from last purchase to reference date; lower is better).
2. **Frequency (F)**: How often do they purchase? (Total count of unique orders; higher is better).
3. **Monetary (M)**: How much money do they spend? (Total currency spent across all orders; higher is better).

### Methodology
- Calculate raw RFM metrics per unique customer (`customer_unique_id`).
- Assign normalized scores on a 1–5 scale using quintile binning (`pd.qcut` and ranking).
- Map RFM score combinations into **9 strategic customer segments** (Champions, Loyal Customers, Potential Loyalists, New Customers, Promising, At Risk, Cannot Lose Them, Lost Customers, Hibernating).
- Produce 8 executive-grade analytical visualizations.
- Generate summary statistics and actionable marketing recommendations.
- Export cleaned segmentation datasets for downstream reporting and dashboard integration.

## 1. Setup & Data Loading
We import required analytical and visualization libraries, configure aesthetic plotting parameters, and load the processed `master_dataset.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Style settings
plt.style.use('seaborn-v0_8-whitegrid')
SNS_PALETTE = 'husl'
FIGURE_SIZE = (12, 6)

# Load master dataset
df = pd.read_csv('../exports/master_dataset.csv', parse_dates=['order_purchase_timestamp'])
print(f'Master dataset shape: {df.shape}')
print(f'Analysis Date Range: {df["order_purchase_timestamp"].min().date()} to {df["order_purchase_timestamp"].max().date()}')
print(f'Total Unique Customers: {df["customer_unique_id"].nunique():,}')
print(f'Total Orders: {df["order_id"].nunique():,}')
print(f'Total Revenue: ${df["payment_value"].sum():,.2f}')
df.head(3)

## 2. RFM Calculation
We establish our **reference snapshot date** as one day after the most recent order timestamp in the dataset. We then aggregate transactions at the `customer_unique_id` level to compute:
- `recency_days`: Difference in days between reference date and the customer's last order.
- `frequency`: Count of distinct orders placed (`nunique` of `order_id`).
- `monetary`: Total monetary spend (`sum` of `payment_value`).

In [ ]:
# Reference date = day after last transaction
ref_date = df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
print(f'Reference Date: {ref_date.date()}')

# Calculate RFM per customer_unique_id
rfm = df.groupby('customer_unique_id').agg(
    recency_days=('order_purchase_timestamp', lambda x: (ref_date - x.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('payment_value', 'sum')
).reset_index()

# Display distributions
print(rfm.describe())

## 3. RFM Score Assignment (1-5 Scale)
We categorize customers into quintiles (5 bins, 20% each) for each RFM dimension:
- **Recency Score (`r_score`)**: Lower recency is better (bought recently). Hence, quintiles are reverse-labeled `[5, 4, 3, 2, 1]`.
- **Frequency Score (`f_score`)**: Higher frequency is better. Because e-commerce order frequency is heavily concentrated at 1 purchase, we use `rank(method='first')` prior to binning to cleanly separate quintiles `[1, 2, 3, 4, 5]`.
- **Monetary Score (`m_score`)**: Higher spend is better. We rank and bin monetary value into quintiles `[1, 2, 3, 4, 5]`.
- **`rfm_score`**: Aggregate score calculated as $R + F + M$ (ranging from 3 to 15).
- **`rfm_string`**: Concatenated string identifier (e.g., `'555'` represents the highest-tier customer).

In [ ]:
# Use pd.qcut for scoring
# Recency: lower is better -> reverse scoring
rfm['r_score'] = pd.qcut(rfm['recency_days'], q=5, labels=[5,4,3,2,1]).astype(int)
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm['m_score'] = pd.qcut(rfm['monetary'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm['rfm_score'] = rfm['r_score'] + rfm['f_score'] + rfm['m_score']
rfm['rfm_string'] = rfm['r_score'].astype(str) + rfm['f_score'].astype(str) + rfm['m_score'].astype(str)

print("RFM Scoring complete. Sample records:")
rfm.head()

## 4. Segment Assignment
We apply business segmentation logic to map the individual R, F, and M scores into 9 actionable segments:
- **Champions**: High recency, high frequency, high spend ($R \ge 4, F \ge 4, M \ge 4$).
- **Loyal Customers**: Consistent repeat shoppers across metrics ($R \ge 3, F \ge 3, M \ge 3$).
- **Potential Loyalists**: Recent shoppers with moderate frequency ($R \ge 3, F \le 2$).
- **New Customers**: Bought very recently for the first time ($R \ge 4, F \le 1$).
- **Promising**: Good recency and above-average frequency/spend ($R \ge 3, F \ge 2, M \ge 2$).
- **At Risk**: High past frequency/spend but haven't purchased recently ($R \le 2, F \ge 3$).
- **Cannot Lose Them**: Lapsed high-value customers needing urgent reactivation ($R \le 2, F \le 2, M \ge 2$).
- **Lost Customers**: Lowest recency and lowest engagement ($R \le 1, F \le 1$).
- **Hibernating**: All other inactive or low-engagement customers.

In [ ]:
def assign_segment(row):
    r, f, m = row['r_score'], row['f_score'], row['m_score']
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3 and m >= 3:
        return 'Loyal Customers'
    elif r >= 3 and f <= 2:
        return 'Potential Loyalists'
    elif r >= 4 and f <= 1:
        return 'New Customers'
    elif r >= 3 and f >= 2 and m >= 2:
        return 'Promising'
    elif r <= 2 and f >= 3:
        return 'At Risk'
    elif r <= 2 and f <= 2 and m >= 2:
        return 'Cannot Lose Them'
    elif r <= 1 and f <= 1:
        return 'Lost Customers'
    else:
        return 'Hibernating'

rfm['segment'] = rfm.apply(assign_segment, axis=1)

print("Customer Count by Segment:")
print(rfm['segment'].value_counts())

## 5. Visualizations & Analytical Explorations
We present 8 executive charts illustrating segment distribution, revenue concentration, score distributions, and multi-dimensional behavioral patterns.

### Figure 1: Segment Customer Count
Horizontal bar chart displaying total customer counts and percentage share per segment.

In [ ]:
# Fig 1: Segment Customer Count (horizontal bar chart, colorful)
plt.figure(figsize=(12, 7))
seg_counts = rfm['segment'].value_counts().sort_values(ascending=True)
colors = sns.color_palette(SNS_PALETTE, len(seg_counts))

bars = plt.barh(seg_counts.index, seg_counts.values, color=colors, edgecolor='black', alpha=0.85)

total_cust = len(rfm)
for bar in bars:
    width = bar.get_width()
    pct = (width / total_cust) * 100
    plt.text(width + total_cust * 0.005, bar.get_y() + bar.get_height() / 2,
             f'{width:,.0f} ({pct:.1f}%)',
             ha='left', va='center', fontsize=11, fontweight='bold')

plt.title('Figure 1: Customer Count by RFM Segment', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Number of Customers', fontsize=12, labelpad=10)
plt.ylabel('RFM Segment', fontsize=12, labelpad=10)
plt.xlim(0, seg_counts.max() * 1.18)
plt.tight_layout()
plt.show()

### Figure 2: Revenue by Customer Segment
Bar chart showing total monetary contribution by segment, sorted descending with percentage annotations.

In [ ]:
# Fig 2: Revenue by Segment (bar chart sorted by revenue)
plt.figure(figsize=(12, 6))
seg_rev = rfm.groupby('segment')['monetary'].sum().sort_values(ascending=False)
colors = sns.color_palette('viridis', len(seg_rev))

bars = plt.bar(seg_rev.index, seg_rev.values, color=colors, edgecolor='black', alpha=0.85)

total_rev = seg_rev.sum()
for bar in bars:
    height = bar.get_height()
    pct = (height / total_rev) * 100
    plt.text(bar.get_x() + bar.get_width() / 2, height + total_rev * 0.008,
             f'${height:,.0f}\n({pct:.1f}%)',
             ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.title('Figure 2: Total Revenue by Customer Segment', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('RFM Segment', fontsize=12, labelpad=10)
plt.ylabel('Total Monetary Value (Revenue)', fontsize=12, labelpad=10)
plt.xticks(rotation=35, ha='right', fontsize=11)
plt.ylim(0, seg_rev.max() * 1.16)
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, loc: f'${x*1e-3:,.0f}K' if x < 1e6 else f'${x*1e-6:,.2f}M'))
plt.tight_layout()
plt.show()

### Figure 3: RFM Score Distribution
Histogram highlighting the distribution of total composite RFM scores ($R + F + M$), along with mean and median markers.

In [ ]:
# Fig 3: RFM Score Distribution (histogram)
fig, ax = plt.subplots(figsize=(12, 6))

score_counts = rfm['rfm_score'].value_counts().sort_index()
norm = plt.Normalize(score_counts.index.min(), score_counts.index.max())
cmap = plt.cm.get_cmap('coolwarm')
bar_colors = [cmap(norm(val)) for val in score_counts.index]

bars = ax.bar(score_counts.index, score_counts.values, color=bar_colors, edgecolor='black', alpha=0.85, width=0.7)

mean_score = rfm['rfm_score'].mean()
median_score = rfm['rfm_score'].median()

ax.axvline(mean_score, color='red', linestyle='--', linewidth=2, label=f'Mean RFM Score: {mean_score:.2f}')
ax.axvline(median_score, color='green', linestyle=':', linewidth=2, label=f'Median RFM Score: {median_score:.0f}')

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, height + len(rfm) * 0.005,
            f'{height:,.0f}', ha='center', va='bottom', fontsize=9.5)

ax.set_title('Figure 3: Overall RFM Score Distribution (Sum of R + F + M)', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Combined RFM Score (Range: 3 to 15)', fontsize=12, labelpad=10)
ax.set_ylabel('Number of Customers', fontsize=12, labelpad=10)
ax.set_xticks(range(3, 16))
ax.legend(frameon=True, fontsize=11)
plt.tight_layout()
plt.show()

### Figure 4: Recency vs Frequency Scatter Plot
Scatter plot examining the relationship between customer recency and order frequency, color-coded by assigned segment.

In [ ]:
# Fig 4: Recency vs Frequency scatter plot (colored by segment)
plt.figure(figsize=(13, 7))

# Add subtle random jitter to frequency to visually distinguish discrete overlapping points
np.random.seed(42)
jitter_freq = rfm['frequency'] + np.random.normal(0, 0.06, size=len(rfm))

sns.scatterplot(
    data=rfm,
    x='recency_days',
    y=jitter_freq,
    hue='segment',
    palette='tab10',
    alpha=0.6,
    s=40,
    edgecolor=None
)

plt.title('Figure 4: Recency vs Frequency by Customer Segment', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Recency (Days Since Last Order)', fontsize=12, labelpad=10)
plt.ylabel('Order Frequency (with subtle visual jitter)', fontsize=12, labelpad=10)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title='RFM Segment', frameon=True)
plt.tight_layout()
plt.show()

### Figure 5: Treemap Representation of Segment Proportions
A hierarchical area treemap displaying segment sizes and percentages using squarified rectangular decomposition.

In [ ]:
# Fig 5: Treemap-style chart of segment sizes
fig, ax = plt.subplots(figsize=(14, 8))

seg_sizes = rfm['segment'].value_counts()
labels = [f"{seg}\n{cnt:,} ({cnt/len(rfm)*100:.1f}%)" for seg, cnt in seg_sizes.items()]
colors = sns.color_palette(SNS_PALETTE, len(seg_sizes))

try:
    import squarify
    squarify.plot(sizes=seg_sizes.values, label=labels, color=colors, alpha=0.85, edgecolor="white", linewidth=2, text_kwargs={'fontsize': 10, 'weight': 'bold'})
    plt.axis('off')
except ImportError:
    # Robust pure matplotlib treemap layout fallback
    import matplotlib.patches as patches
    def squarify_layout(values, x, y, dx, dy):
        if not values:
            return []
        if len(values) == 1:
            return [{'x': x, 'y': y, 'dx': dx, 'dy': dy}]
        mid = len(values) // 2
        s1 = sum(values[:mid])
        s2 = sum(values[mid:])
        total = s1 + s2
        if total == 0:
            return []
        if dx >= dy:
            dx1 = dx * (s1 / total)
            dx2 = dx - dx1
            r1 = squarify_layout(values[:mid], x, y, dx1, dy)
            r2 = squarify_layout(values[mid:], x + dx1, y, dx2, dy)
        else:
            dy1 = dy * (s1 / total)
            dy2 = dy - dy1
            r1 = squarify_layout(values[:mid], x, y, dx, dy1)
            r2 = squarify_layout(values[mid:], x, y + dy1, dx, dy2)
        return r1 + r2

    boxes = squarify_layout(list(seg_sizes.values), 0, 0, 100, 100)
    for box, label, color in zip(boxes, labels, colors):
        rect = patches.Rectangle((box['x'], box['y']), box['dx'], box['dy'], linewidth=2, edgecolor='white', facecolor=color, alpha=0.85)
        ax.add_patch(rect)
        cx = box['x'] + box['dx'] / 2
        cy = box['y'] + box['dy'] / 2
        ax.text(cx, cy, label, ha='center', va='center', fontsize=9.5, fontweight='bold', color='black')
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
    plt.axis('off')

plt.title('Figure 5: Treemap Representation of RFM Segment Proportions', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

### Figure 6: Average Recency, Frequency, and Monetary Value by Segment
Comparative horizontal panel bars showcasing the average raw metric values for each customer segment.

In [ ]:
# Fig 6: Average R, F, M by segment (grouped multi-panel chart)
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=False)

seg_metrics = rfm.groupby('segment').agg(
    avg_recency=('recency_days', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean')
).reindex(rfm['segment'].value_counts().index)

colors = sns.color_palette(SNS_PALETTE, len(seg_metrics))

# 1. Average Recency
axes[0].barh(seg_metrics.index, seg_metrics['avg_recency'], color=colors, edgecolor='black', alpha=0.85)
axes[0].set_title('Avg Recency (Days - Lower is Better)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Days')
for i, v in enumerate(seg_metrics['avg_recency']):
    axes[0].text(v + 5, i, f'{v:.1f}d', va='center', fontweight='bold', fontsize=10)

# 2. Average Frequency
axes[1].barh(seg_metrics.index, seg_metrics['avg_frequency'], color=colors, edgecolor='black', alpha=0.85)
axes[1].set_title('Avg Order Frequency (Higher is Better)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Orders')
for i, v in enumerate(seg_metrics['avg_frequency']):
    axes[1].text(v + 0.03, i, f'{v:.2f}', va='center', fontweight='bold', fontsize=10)

# 3. Average Monetary
axes[2].barh(seg_metrics.index, seg_metrics['avg_monetary'], color=colors, edgecolor='black', alpha=0.85)
axes[2].set_title('Avg Monetary Value (Higher is Better)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Monetary ($)')
for i, v in enumerate(seg_metrics['avg_monetary']):
    axes[2].text(v + 5, i, f'${v:,.1f}', va='center', fontweight='bold', fontsize=10)

plt.suptitle('Figure 6: Average Recency, Frequency, and Monetary Metrics by Segment', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Figure 7: Monetary Distribution by Segment (Boxplot)
Log-scale boxplot visualizing the spend distribution across segments, highlighting medians, interquartile ranges, and mean spend (red markers).

In [ ]:
# Fig 7: Monetary distribution by segment (boxplot)
plt.figure(figsize=(14, 7))

sns.boxplot(
    data=rfm,
    x='segment',
    y='monetary',
    palette=SNS_PALETTE,
    showmeans=True,
    meanprops={"marker":"o", "markerfacecolor":"red", "markeredgecolor":"black", "markersize":"8"}
)

plt.title('Figure 7: Monetary Spend Distribution by Segment (Log Scale, Red Dot = Mean)', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Customer Segment', fontsize=12, labelpad=10)
plt.ylabel('Monetary Value ($) [Log Scale]', fontsize=12, labelpad=10)
plt.yscale('log')
plt.xticks(rotation=35, ha='right', fontsize=11)
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'${y:g}'))
plt.tight_layout()
plt.show()

### Figure 8: Heatmap of Average RFM Values by Segment
Normalized performance heatmap illustrating the relative strengths and vulnerabilities of each customer cohort across the R, F, and M dimensions.

In [ ]:
# Fig 8: Heatmap of avg RFM values by segment
plt.figure(figsize=(11, 6.5))

heatmap_data = rfm.groupby('segment')[['recency_days', 'frequency', 'monetary']].mean()

# Normalize metrics (MinMax scaling) to a unified 0-1 scale
heatmap_normalized = (heatmap_data - heatmap_data.min()) / (heatmap_data.max() - heatmap_data.min())
# Invert recency normalization so 1 = best (most recent) and 0 = worst (least recent)
heatmap_normalized['recency_days'] = 1 - heatmap_normalized['recency_days']
heatmap_normalized.columns = ['Recency (Inverted: 1=Recent)', 'Frequency (1=High)', 'Monetary (1=High)']

sns.heatmap(
    heatmap_normalized,
    annot=heatmap_data.values,
    fmt='.1f',
    cmap='YlGnBu',
    linewidths=1.5,
    cbar_kws={'label': 'Normalized Relative Strength (0 = Low, 1 = High)'}
)

plt.title('Figure 8: RFM Segment Heatmap (Color: Relative Score | Numbers: Actual Averages)', fontsize=15, fontweight='bold', pad=15)
plt.ylabel('Customer Segment', fontsize=12)
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)
plt.tight_layout()
plt.show()

## 6. Segment Summary Statistics Table
We aggregate segment-level metrics to establish a unified performance baseline: customer headcount, customer proportion, average recency, average frequency, average spend, and total generated revenue.

In [ ]:
segment_summary = rfm.groupby('segment').agg(
    customer_count=('customer_unique_id', 'count'),
    pct_customers=('customer_unique_id', lambda x: round(len(x)/len(rfm)*100, 1)),
    avg_recency=('recency_days', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean'),
    total_revenue=('monetary', 'sum')
).reset_index().sort_values('total_revenue', ascending=False)

segment_summary['pct_revenue'] = (segment_summary['total_revenue'] / segment_summary['total_revenue'].sum() * 100).round(1)

print(segment_summary.to_string(index=False))

## 7. Business Interpretation & Strategic Marketing Playbook

The RFM segmentation reveals profound behavioral distinctions across our customer base. Below is our operational strategy and targeted marketing action plan for each segment:

| Segment | Behavioral Profile | Strategic Objective | Actionable Marketing Tactics |
| :--- | :--- | :--- | :--- |
| **Champions** | Bought very recently, buy frequently, and spend the most. | Maximize lifetime value, retain at all costs, cultivate brand advocates. | - VIP concierge support and early access to product launches.<br>- Invite-only loyalty tier with exclusive rewards.<br>- Request reviews, user-generated content, and referral invites. |
| **Loyal Customers** | Consistent purchase cadence, above-average spend, and responsive to offers. | Increase average order value (AOV) and incentivize cross-category discovery. | - Personalized bundle recommendations based on past purchases.<br>- Tiered threshold promotions (e.g., *Spend $150, Get $25 Gift Card*).<br>- Loyalty points accelerators on high-margin product categories. |
| **Potential Loyalists** | Recent buyers with average frequency and solid spend potential. | Convert into repeat, habitual shoppers. | - 30-day post-purchase automated email sequence with complementary accessories.<br>- Membership program invitations highlighting shipping perks.<br>- Time-delimited discount for a 2nd order placed within 21 days. |
| **New Customers** | High recency score but single-order history. Low familiarity with brand breadth. | Accelerate onboarding and prompt the critical second purchase. | - Welcoming educational email series featuring customer favorites and bestsellers.<br>- First-time review incentive accompanied by a bounce-back voucher.<br>- Frictionless customer service follow-up to ensure fulfillment satisfaction. |
| **Promising** | Recent shoppers with moderate spend and engagement; high upside. | Deepen brand affinity and increase purchase frequency. | - Targeted category cross-sells based on collaborative filtering algorithms.<br>- Limited-time free shipping passes or flash sale notifications.<br>- Social proof campaigns showcasing real customer testimonials. |
| **At Risk** | Formerly frequent and high-value buyers who have not purchased in a long period. | Re-engage before complete churn and reclaim market share. | - Automated win-back email sequence with increasing discount incentives (10% $\rightarrow$ 20%).<br>- Feedback survey: *"We miss you – tell us how we can improve."*<br>- High-priority retargeting on social media highlighting newly added catalog items. |
| **Cannot Lose Them** | High historical spenders with critically low recency scores. Major revenue impact. | Aggressive reactivation through personalized multi-channel outreach. | - Direct VIP customer outreach from senior account managers or customer success.<br>- High-value, personalized win-back offers with no minimum spend.<br>- Detailed diagnostic of past service or delivery complaints to address root dissatisfaction. |
| **Lost Customers** | Lowest recency and lowest engagement. Lapsed over a year ago with low spend. | Low-cost automated win-back or minimize marketing waste. | - Automated quarterly re-engagement blast with aggressive clearance pricing.<br>- Exclude from high-cost paid ad targeting to optimize marketing return on ad spend (ROAS).<br>- Sunset inactive email subscribers after 90 days of non-responsiveness to protect domain reputation. |
| **Hibernating** | Low frequency, low spend, and long periods of inactivity. | Selective reactivation during peak shopping holidays. | - Include in Black Friday, Cyber Monday, and Semi-Annual clearance blasts.<br>- Lightweight SMS or push notifications promoting clearance flash sales.<br>- Move to low-frequency email cadences to prevent unsubscribes. |

## 8. Export Cleaned Segment Datasets
We export the processed customer-level segment table (`rfm_segments.csv`) and segment summary table (`segment_summary.csv`) to the `../exports/` directory for consumption by downstream retention models, executive reports, and Power BI dashboards.

In [ ]:
import os
os.makedirs('../exports', exist_ok=True)

rfm.to_csv('../exports/rfm_segments.csv', index=False)
segment_summary.to_csv('../exports/segment_summary.csv', index=False)

print("RFM analysis datasets exported successfully to '../exports/':")
print(f" - ../exports/rfm_segments.csv ({len(rfm):,} rows)")
print(f" - ../exports/segment_summary.csv ({len(segment_summary):,} rows)")